# Vector compression — token study + Canvas PDF retrieval

**Survey:** JL projections, rank-k SVD, sign/scalar quantization — see `docs/SURVEY.md`.

**Part A:** 230 word embeddings (Azure `text-embedding-3-small`, d=256); nearest-neighbor recall under compression.

**Part B (main application):** Sync **MATH 5110 course PDFs from Canvas**, chunk text, embed chunks, compress the **retrieval index**, measure hit@k on labeled queries.

Requires `.env` with `AZURE_OPENAI_*` and `CANVAS_*` (see `.env.example`). First run downloads PDFs and caches embeddings to `python/data/*.parquet`.

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

import polars as pl
from IPython.display import display, Markdown


def find_repo() -> Path:
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise RuntimeError("Open this notebook from the repo (e.g. python/notebooks/)")


REPO = find_repo()
sys.path.insert(0, str(REPO / "python" / "src"))

from vector_linalg.config import load_config
from vector_linalg.embeddings import embedding_matrix, fetch_token_embeddings
from vector_linalg.metrics import results_table
from vector_linalg.plots import (
    figure_distance_error,
    figure_rag_compare,
    figure_recall_vs_bits,
    figure_token_pca,
)
from vector_linalg.rag import fetch_rag_embeddings, run_rag_compression_study
from vector_linalg.study import run_compression_study

cfg = load_config()
provider = cfg.embeddings.provider
display(Markdown(f"Repo: `{cfg.repo_root}` | embeddings: **{provider}** | RAG source: **{cfg.rag.source}**"))

## Part A — Token embedding compression (toy universe)

In [ ]:
df = fetch_token_embeddings(cfg)
tokens, keys = embedding_matrix(df)
dim = keys.shape[1]
display(Markdown(f"**{len(tokens)}** tokens, **d={dim}** (`{cfg.embeddings.model}` via {provider})"))
display(df.select(["token", "d0", "d1", "d2", "d3"]).head(8))

In [ ]:
token_results = run_compression_study(keys, cfg)
display(results_table(token_results, cfg.recall_k))

In [ ]:
display(figure_token_pca(keys, tokens, highlight=["king", "queen", "matrix", "vector", "eigenvalue"]))
display(figure_recall_vs_bits(token_results, cfg.recall_k, title="Token recall vs compression"))
display(figure_distance_error(token_results))

## Part B — RAG on Canvas course PDFs

Pipeline (same as `scripts/run_all.py`):
1. Scrape linked `/files/<id>` URLs from Canvas module pages
2. Download PDFs → PyMuPDF text → ~900-char chunks
3. Embed all chunks (the **search index**)
4. Compress the index; score queries with hit@k

If `rag_queries.yaml` labels are stale, synthetic queries are built from sample chunk text.

In [ ]:
if not cfg.rag.enabled:
    display(Markdown("RAG disabled in config.yaml"))
else:
    rag_bundle = fetch_rag_embeddings(cfg)
    n_chunks = len(rag_bundle.chunk_ids)
    n_queries = len(rag_bundle.query_texts)
    display(Markdown(f"**{n_chunks}** chunks from Canvas PDFs, **{n_queries}** eval queries"))

    sample = pl.DataFrame({
        "chunk_id": rag_bundle.chunk_ids[:5],
        "preview": [t[:120] + "..." for t in rag_bundle.chunk_texts[:5]],
    })
    display(sample)

In [ ]:
if cfg.rag.enabled:
    rag_results = run_rag_compression_study(rag_bundle, cfg)
    display(results_table(rag_results, cfg.rag.recall_k))

    display(figure_recall_vs_bits(
        rag_results,
        cfg.rag.recall_k,
        title=f"RAG hit@{cfg.rag.recall_k} vs compression (Canvas PDF index)",
    ))
    display(figure_rag_compare(
        token_results,
        rag_results,
        token_k=cfg.recall_k,
        rag_k=cfg.rag.recall_k,
    ))